# 5.4 章节实践：独立补全并验证Matmul + Bias

5.2介绍了一个结果块怎样由Cube和Vector协作完成，5.3让同一个kernel处理不同的行数。本节把这两部分组合起来，由你补全关键代码，再检查它是否满足要求。

主练习始终计算 `Y = A @ B.T + Bias`，依次完成**补全计算、组合动态行数、验证结果**三个任务。

<small>线上环境没有昇腾950，可对照本页代码和实测结果完成练习；自行运行请按1.4准备环境，并以本章目录为工作目录。Notebook用户从下方准备单元开始顺序执行。</small>

<small>下方为运行辅助代码，阅读时可先看练习要求。</small>

In [1]:
from pathlib import Path
import re
import subprocess
import sys

Path("src").mkdir(exist_ok=True)
Path("answer").mkdir(exist_ok=True)

def run_and_show(command):
    # NPU程序在独立进程中运行；当前Notebook内核不导入torch_npu。
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    # 只省略安装目录；数值、校验结果和报错内容保持原样。
    output = result.stdout.replace(str(Path.cwd()), ".")
    output = re.sub(r"/home/[^\s]+", "<已省略安装路径>", output)
    print(output, end="")
    result.check_returncode()
    return result


## 练习要求

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">项目</th>
<th style="text-align: left; vertical-align: top;">本节要求</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">输入与输出</td>
<td style="text-align: left; vertical-align: top;">A为<code>(M, 128)</code>，B为<code>(128, 128)</code>，Bias为<code>(128,)</code>，Y为<code>(M, 128)</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">数据类型</td>
<td style="text-align: left; vertical-align: top;">A、B为float16；Bias、累加结果和Y为float32</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">分块</td>
<td style="text-align: left; vertical-align: top;"><code>BM = BN = BK = 64</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">行数范围</td>
<td style="text-align: left; vertical-align: top;">当前教学版只处理完整块，支持M为64的正整数倍；暂不处理尾块</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">编译与调用</td>
<td style="text-align: left; vertical-align: top;">编译一次，再用同一个kernel处理不同M</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">结果检查</td>
<td style="text-align: left; vertical-align: top;">比较完整输出，并在调用前拒绝不符合要求的输入</td>
</tr>
</tbody>
</table>

<small>练习建议：先根据数据位置和形状推导；遇到困难时查看对应提示，完成后再核对答案。</small>

## 任务一：补全一个结果块的计算

下面省略已学过的参数声明和局部空间分配，只留下需要你补全的计算部分。已知`row`、`col`是当前结果块的编号，`a`和`b`位于L1，`acc`位于L0C，`v`和`bias`位于各Vector侧的UB。

将 `__1__` 到 `__6__` 替换为相应表达式。

<small>题目说明：这些名字只是填空占位符，不是TileLang API。</small>

```python
for ki in T.serial(K // BK):
    T.copy(A[row * BM:(row + 1) * BM,
             ki * BK:(ki + 1) * BK], a)
    T.copy(B[col * BN:(col + 1) * BN,
             ki * BK:(ki + 1) * BK], b)
    T.gemm(a, b, acc,
           transpose_B=True, clear_accum=(__1__))

T.dual_copy(__2__, __3__)
T.copy(Bias[__4__:__5__], bias)

with T.SimdVF():
    full = S.pset(32, "PAT_ALL")
    for i in T.serial(BM // 2):
        value = S.vld(v[i, 0])
        bv = S.vld(bias[0])
        result = __6__
        S.vsts(v[i, 0], result, full)

T.dual_copy(v, Y[row * BM:(row + 1) * BM,
                 col * BN:(col + 1) * BN])
```

补全后，用自己的话说明：

- 第一轮K计算与后续轮次，对累加器的处理有什么区别？
- 64×64的矩阵乘结果从哪里交给Vector？
- 偏置范围应由row还是col决定？

<details>
<summary>需要帮助时，展开提示</summary>

先想清楚“当前结果块需要哪64个列偏置”。两个Vector侧处理不同的行，但它们负责相同的64列。

</details>

## 任务二：补全动态M，并复用编译结果

沿用任务一的计算部分，在完整kernel的相应位置补全下面三处。A和Y的参数形状都使用同一个M。

```python
# 放在kernel定义之前
M = __7__

# 放在kernel函数开头
T.assume(__8__)
T.assume(__9__)
```

接着检查下面的Host片段。假定B和Bias已经准备好，`check_inputs`会检查输入条件。**这段代码即使数值结果正确，也没有完成本节的编译复用目标。** 请调整它的结构。

```python
for m in (64, 128, 192):
    kernel = tilelang.compile(
        matmul_bias_dynamic, target="ascend", out_idx=-1
    )
    a_cpu = torch.ones((m, K), dtype=torch.float16)
    check_inputs(a_cpu, b_cpu, bias_cpu)
    actual = kernel(a_cpu.npu(), b_npu, bias_npu).cpu()
```

最后，补全Host侧对M的判断：哪些行数可以进入kernel，哪些必须在调用前拒绝？重点检查M=193。仅在DSL中写下`T.assume`，不能代替这一步。

<details>
<summary>需要帮助时，展开提示</summary>

将编译语句与“每次准备输入并调用”的步骤分开。M来自A的实际行数，局部块大小仍为64×64。

</details>

## 任务三：检查测试是否足够

假设程序用全零输入测试通过，能否据此确认偏置索引和写回位置都正确？请先回答，再对照下面的测试安排。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">输入类型</th>
<th style="text-align: left; vertical-align: top;">主要检查什么</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">随机输入</td>
<td style="text-align: left; vertical-align: top;">一般数值是否与PyTorch结果一致</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">全零输入</td>
<td style="text-align: left; vertical-align: top;">零值情况下是否出现异常结果</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">非零常量输入</td>
<td style="text-align: left; vertical-align: top;">累加与偏置相加是否明显缺失</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">按行列位置构造的输入</td>
<td style="text-align: left; vertical-align: top;">行块、列偏置和写回位置是否对应</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">含较大正负值的输入</td>
<td style="text-align: left; vertical-align: top;">更大数值范围内是否仍满足误差要求</td>
</tr>
</tbody>
</table>

用M=64、128、192、256、640分别搭配这五类输入，共检查25组。每组都比较全部`M×128`个元素；选取几个位置打印，只用于帮助读懂结果。

针对当前示例的输入约束，再用M=0、63、65、193、257检查调用前的拒绝逻辑。预期结果是Host报告输入不符合要求。程序不能把其他计算异常也当作“输入检查通过”。

## 参考答案与实际结果

<details>
<summary>展开关键改动与解题说明</summary>

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">空白</th>
<th style="text-align: left; vertical-align: top;">应填内容</th>
<th style="text-align: left; vertical-align: top;">核对理由</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>__1__</code></td>
<td style="text-align: left; vertical-align: top;"><code>ki == 0</code></td>
<td style="text-align: left; vertical-align: top;">第一轮清空累加器，后续轮次继续累加</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>__2__</code>、<code>__3__</code></td>
<td style="text-align: left; vertical-align: top;"><code>acc</code>、<code>v</code></td>
<td style="text-align: left; vertical-align: top;">将L0C结果传到两侧UB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>__4__</code>、<code>__5__</code></td>
<td style="text-align: left; vertical-align: top;"><code>col * BN</code>、<code>(col + 1) * BN</code></td>
<td style="text-align: left; vertical-align: top;">按结果块的列范围读取偏置</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>__6__</code></td>
<td style="text-align: left; vertical-align: top;"><code>S.vadd(value, bv, full)</code></td>
<td style="text-align: left; vertical-align: top;">给当前行的各列加上对应偏置</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>__7__</code></td>
<td style="text-align: left; vertical-align: top;"><code>T.dynamic("num_rows")</code></td>
<td style="text-align: left; vertical-align: top;">M在调用时由输入形状确定</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>__8__</code>、<code>__9__</code></td>
<td style="text-align: left; vertical-align: top;"><code>M &gt; 0</code>、<code>M % BM == 0</code></td>
<td style="text-align: left; vertical-align: top;">将调用方保证的行数条件告知编译器</td>
</tr>
</tbody>
</table>

任务二还需要把`tilelang.compile`移到循环外。Host先取得`m = a.shape[0]`，在`m <= 0 or m % BM != 0`时拒绝输入，再执行设备调用。

任务三不能只用全零输入：如果输出位置放错，而应写入的值恰好都为零，错误就可能没有暴露。按位置构造的输入能帮助检查索引，但同样需要与其他输入类型及完整输出比较配合。

</details>

### 完整程序与运行结果

程序中，`make_inputs`只负责生成上表中的五类输入；`check_inputs`在设备调用之前检查条件。最后两个循环分别检查合法输入的计算结果，以及不符合要求的行数是否被拒绝。

<small>完整参考程序可按需展开，包含kernel、输入构造、输入检查和完整比较。</small>

In [2]:
%%writefile answer/matmul_bias_practice.py
import tilelang
import tilelang.language as T
from tilelang.language import simd as S

tilelang.disable_cache()

N = K = 128
BM = BN = BK = 64
M = T.dynamic("num_rows")  # ① M是运行时行数

@T.prim_func
def matmul_bias_dynamic(
    A: T.Tensor((M, K), "float16"),
    B: T.Tensor((N, K), "float16"),
    Bias: T.Tensor((N,), "float32"),
    Y: T.Tensor((M, N), "float32"),
):
    T.assume(M > 0)  # ② 调用方保证的条件
    T.assume(M % BM == 0)

    # ③ 任务数根据本次M计算
    with T.Kernel((M // BM) * (N // BN)) as bx:
        row = bx // (N // BN)
        col = bx % (N // BN)

        a = T.alloc_l1((BM, BK), "float16")
        b = T.alloc_l1((BN, BK), "float16")
        acc = T.alloc_l0c((BM, BN), "float32")
        v = T.alloc_shared((BM // 2, BN), "float32")
        bias = T.alloc_shared((BN,), "float32")

        for ki in T.serial(K // BK):
            T.copy(A[row * BM:(row + 1) * BM,
                     ki * BK:(ki + 1) * BK], a)
            T.copy(B[col * BN:(col + 1) * BN,
                     ki * BK:(ki + 1) * BK], b)
            T.gemm(a, b, acc,
                   transpose_B=True, clear_accum=(ki == 0))

        T.dual_copy(acc, v)
        T.copy(Bias[col * BN:(col + 1) * BN], bias)

        with T.SimdVF():
            full = S.pset(32, "PAT_ALL")
            for i in T.serial(BM // 2):
                value = S.vld(v[i, 0])
                bv = S.vld(bias[0])
                result = S.vadd(value, bv, full)
                S.vsts(v[i, 0], result, full)

        T.dual_copy(v, Y[row * BM:(row + 1) * BM,
                         col * BN:(col + 1) * BN])

import torch
import torch_npu

def check_inputs(a, b, bias):
    if a.ndim != 2 or b.shape != (N, K) or bias.shape != (N,):
        raise ValueError("输入形状不符合要求")
    if a.shape[1] != K:
        raise ValueError("A的列数必须等于K")
    m = a.shape[0]
    if m <= 0 or m % BM != 0:
        raise ValueError("M必须为64的正整数倍")
    if a.dtype != torch.float16 or b.dtype != torch.float16 or bias.dtype != torch.float32:
        raise ValueError("A、B必须为float16，Bias必须为float32")
    if not all(x.is_contiguous() for x in (a, b, bias)):
        raise ValueError("输入必须连续存放")
    return m

def make_inputs(m, kind):
    if kind == "random":
        a = (torch.randn(m, K) * 0.125).half()
        b = (torch.randn(N, K) * 0.125).half()
        bias = torch.randn(N)
    elif kind == "zero":
        a = torch.zeros((m, K), dtype=torch.float16)
        b = torch.zeros((N, K), dtype=torch.float16)
        bias = torch.zeros(N)
    elif kind == "constant":
        a = torch.full((m, K), 0.125, dtype=torch.float16)
        b = torch.full((N, K), -0.25, dtype=torch.float16)
        bias = torch.full((N,), 0.5)
    elif kind == "indexed":
        a = ((torch.arange(m)[:, None] % 7 - 3).expand(m, K).float() / 16).half().contiguous()
        b = ((torch.arange(N)[:, None] % 5 - 2).expand(N, K).float() / 8).half().contiguous()
        bias = torch.arange(N).float() / 16
    elif kind == "signed":
        a = (torch.randn(m, K) * 2).half()
        b = (torch.randn(N, K) * 2).half()
        bias = torch.linspace(-4, 4, N)
    else:
        raise ValueError("未知的测试类型")
    return a, b, bias

torch.npu.set_device(0)
torch.manual_seed(20260915)
kernel = tilelang.compile(matmul_bias_dynamic, target="ascend", out_idx=-1)

results = []
for m in (64, 128, 192, 256, 640):
    for kind in ("random", "zero", "constant", "indexed", "signed"):
        a_cpu, b_cpu, bias_cpu = make_inputs(m, kind)
        check_inputs(a_cpu, b_cpu, bias_cpu)
        actual = kernel(a_cpu.npu(), b_cpu.npu(), bias_cpu.npu()).cpu()
        expected = a_cpu.float() @ b_cpu.float().T + bias_cpu
        torch.testing.assert_close(actual, expected, rtol=0.002, atol=0.002)
        max_error = (actual - expected).abs().max().item()
        results.append({"M": m, "case": kind, "elements": actual.numel(),
                        "max_abs_error": max_error, "passed": True})
        if m == 64 and kind == "indexed":
            points = [(0, 0), (31, 63), (32, 64), (63, 127)]
            print("按位置构造的输入，四个检查点：",
                  [actual[r, c].item() for r, c in points])
    print(f"M={m}：5类输入的完整输出比较均通过")

rejected = []
for m in (0, 63, 65, 193, 257):
    a_cpu = torch.ones((m, K), dtype=torch.float16)
    b_cpu = torch.ones((N, K), dtype=torch.float16)
    bias_cpu = torch.zeros(N)
    try:
        check_inputs(a_cpu, b_cpu, bias_cpu)
    except ValueError as error:
        rejected.append({"M": m, "error": str(error)})
        print(f"M={m}：调用前拒绝，{error}")
    else:
        raise AssertionError(f"M={m}应当被拒绝")

print(f"Verification passed! {len(results)}组完整输出比较通过，"
      f"{len(rejected)}种不符合要求的行数均在调用前拒绝。")


Overwriting answer/matmul_bias_practice.py


**运行与检查结果**

在本章目录的终端中运行，或执行下方Notebook单元：

```bash
python answer/matmul_bias_practice.py
```

In [3]:
run_result = run_and_show([sys.executable, "-u", 'answer/matmul_bias_practice.py'])


Loading tilelang libs from dev root: <已省略安装路径>
[W918 20:21:17.498215615 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 20:21:25  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `matmul_bias_dynamic` with `out_idx=-1`
2026-09-18 20:21:37  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `matmul_bias_dynamic`
按位置构造的输入，四个检查点： [6.0, 3.9375, 6.0, 7.9375]
M=64：5类输入的完整输出比较均通过
M=128：5类输入的完整输出比较均通过
M=192：5类输入的完整输出比较均通过
M=256：5类输入的完整输出比较均通过
M=640：5类输入的完整输出比较均通过
M=0：调用前拒绝，M必须为64的正整数倍
M=63：调用前拒绝，M必须为64的正整数倍
M=65：调用前拒绝，M必须为64的正整数倍
M=193：调用前拒绝，M必须为64的正整数倍
M=257：调用前拒绝，M必须为64的正整数倍
Verification passed! 25组完整输出比较通过，5种不符合要求的行数均在调用前拒绝。


**怎样判断运行结果**

重点查看两类结果：**25组合法输入是否全部通过完整输出比较**，以及**5种不符合要求的行数是否都在调用前被拒绝**。程序打印的四个位置只展示了部分结果，不能代替完整输出比较。

检查自己的答案时，可以逐项确认：是否保留了整个K维度的累加，是否按col选取Bias，是否只执行了一次编译，以及M=193是否在调用前被拒绝。

## 本章回顾与后续练习

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">知识点</th>
<th style="text-align: left; vertical-align: top;">完成本章后应能做到</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">Cube与Vector协作</td>
<td style="text-align: left; vertical-align: top;">将矩阵乘与偏置相加对应到C核、V核</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">列偏置</td>
<td style="text-align: left; vertical-align: top;">根据结果块的列编号选择Bias范围</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">K维累加</td>
<td style="text-align: left; vertical-align: top;">第一轮清空，后续轮次继续累加</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">动态M</td>
<td style="text-align: left; vertical-align: top;">根据实际行数确定任务数量和访问范围</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">固定局部块</td>
<td style="text-align: left; vertical-align: top;">区分总行数变化与局部空间大小</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">Host检查与assume</td>
<td style="text-align: left; vertical-align: top;">先检查输入，再向编译器声明成立的条件</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">完整结果比较</td>
<td style="text-align: left; vertical-align: top;">结合不同输入类型检查全部输出元素</td>
</tr>
</tbody>
</table>

本节完成了从补全代码到验证结果的练习。接下来，第6章将介绍**怎样定位数值错误，以及怎样测量算子耗时**。

<details>
<summary>选做：继续尝试其他场景</summary>

完成主练习后，还可以选择两个拓展方向：

- **向量计算迁移：** 尝试实现`Y = SiLU(A @ B.T)`。这需要另外学习SiLU公式及对应的指数、除法和分支选择指令。
- **性能实验：** 学习第6章的计时方法后，改变一个已理解的参数，再用相同条件比较耗时。

</details>

[章节导航](README.md) · [上一节](05.03_dynamic_shapes.ipynb) · [下一章](../06_tuning/06.01_introduction.ipynb)